# report1 — Sionna 환경 최적화 + 실물 3D 메쉬 + 분절 드론

> ### ❓ 이 리포트가 답하는 질문
> **Sionna 안에 무엇을 어떻게 세웠고, 그 드론 모델을 믿어도 되는가?**

### ⚡ 결론부터 (TL;DR)

1. **챔버는 semi-anechoic 이다.** 30×20×11 m, 벽4+천장은 피라미드 흡수체지만 **바닥은 반사성 콘크리트(ITU)**. 라디오맵에 바닥이 그대로 살아 있고, **드론이 그림자를 드리운다**. → **"이 방은 무반사라 클러터가 약하다"는 폐기된 서사입니다** (바닥에는 성립하지 않는다).
2. **메쉬는 실물 CAD 다.** trimesh + **manifold3d(불리언 CSG)** + shapely + scipy 로 5종을 다시 만들었다(삼각형 24,398~39,158). 5종 전부 DJI 공식 외형과 **0.00 % 일치**하고, trimesh 검증 **5/5 통과**.
3. **불리언이 없으면 PO 가 헛센다** — 겹친 파트의 내부 면 **1374개**(면적의 12.9 %)가 드론 **속에** 살아 있었고, 가림 없는 PO 는 그것을 그대로 적분해 σ 를 **+0.77 dB** 부풀렸다.
4. **대각선은 지어낼 뻔했다.** Mavic 4 Pro 의 '400 mm'는 출처가 없고 267 mm 프롭과 **기하학적으로 모순**이다(앞뒤 디스크가 14 mm 겹친다). 공식 외형에서 유도하면 **439 mm**. Matrice 4E 는 외형과 대각선을 **둘 다** 공개한 유일 기종인데, **외형만 맞춘** 우리 메쉬의 대각선이 431 mm (-1.8 %) → **방법론 독립 검증**.
5. **호버 RPM 을 물리로 유도했다** ($T = C_T\rho n^2 D^4$). 그 결과 mavic4pro 는 5500 → **3600 rpm** 으로 정정됐다 (옛 5500 은 중량의 **2.3 배** 추력 = 최대추력 회전수였다). → flash 183 → **120 Hz**, f_tip 1734 → **1135 Hz**. **옛 숫자를 쓰지 말 것.**
6. **분절 모델이 마이크로도플러와 Gazebo 의 공통 토대다.** 몸체 자세와 로터 스핀이 완전히 분리된다(블레이드만 90° 돌려도 프레임 정점 이동 = 0.0e+00 m). 가림을 넣은 SBR 로 5종 스펙트로그램을 냈고, 옛 순수 PO 대비 |DC|/std(AC) 가 **8~36 dB** 낮다(= 검출이 그만큼 쉽다).

### 🗺️ 어디부터 읽나

| 절 | 무엇을 |  |
|---|---|---|
| §1 | 챔버 — Sionna 안의 실험장 | 재질 표 · Sionna 렌더 · 라디오맵(드론 그림자) |
| §2 | 드론 메쉬 — **어떻게 만들었나** | 자료조사→적대적 검증→CAD→불리언→검증→Sionna 갤러리 |
| §3 | 분절 — 마이크로도플러 & Gazebo | 호버 RPM 유도 · SBR 스펙트로그램 · PX4 에 뭐가 더 필요한가 |
| §4 | 정리 & 다음 | report2/3 로 무엇이 넘어가나 |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 드론 5종 제원 (외형·프롭·질량·RPM·재질·PX4 파라미터) | `docs/drone_specs_2026.json` — DJI 공식 스펙 페이지 + 사용자 매뉴얼 PDF 직접 파싱(PyMuPDF) + 티어다운. **출처 URL 이 기종별로 다 들어 있다.** | 1차 자료 + **적대적 재검증**(별도 에이전트가 '틀렸다고 가정하고' 재확인 → 지어낸 숫자 3건 적발) |
| 재질 (εr, σ) | ITU-R P.2040 (Sionna 내장). 없는 재질만 문헌값 | 표준 — **Sionna 에게 물어본다**(우리가 안 적는다) |
| 챔버 치수·구성 | 사용자가 준 대형 차폐시설 사진 + `src/chamber.py` | 모델(30×20×11 m, semi-anechoic) |
| 호버 RPM | $T = C_T \rho n^2 D^4$ (운동량/블레이드요소 이론의 표준 추력계수 형태), $C_T$ = 0.10~0.12 | **유도값** — DJI 는 호버 RPM 을 공개하지 않는다 |
| Gazebo/PX4 파라미터 | `docs/drone_specs_2026.json` 의 `gazebo_px4` 필드 | 공개값 + **추정값(명시)** — 관성텐서·k_M·시상수는 어느 기종도 미공개 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-render` | Sionna RT `Scene.render_to_file()` — 씬·**추적된 광선**·라디오맵을 사진처럼 렌더 | 🟢 **Sionna 내부** (Mitsuba 3 경로추적 렌더러, GPU) |
| `sionna-radiomap` | Sionna RT `RadioMapSolver` — 공간별 전파 세기 분포 | 🟢 **Sionna 내부** (GPU) |
| `sionna-rt` | Sionna RT `PathSolver` — 전파 광선추적. 경로별 **지연 τ · 도플러 f_d · 복소이득 · 반사점 좌표**를 준다 | 🟢 **Sionna 내부** (Mitsuba 3 / OptiX, GPU) |
| `trimesh-cad` | CAD 모델링 (`src/cadkit.py` + `src/drone_cad.py`) — 로프트·스윕·회전체·**불리언(CSG)** | 🔴 **별도** (trimesh + manifold3d + shapely + scipy, CPU) |
| `trimesh-check` | 메쉬 검증 (`src/mesh_check.py`) — watertight · winding · 법선방향 · 퇴화면 | 🔴 **별도** (trimesh, CPU). 빌드 게이트로 회귀를 막는다 |
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `microdoppler` | 마이크로도플러 (`src/microdoppler.py`) — 회전 블레이드의 슬로타임 복소장 → STFT | 🟡 **우리가 짰다** — 자세별 산란장은 SBR(Mitsuba 광선)로 계산 (GPU) |
| `po` | 순수 물리광학 (`src/rcs_po.py`) — 점구름 PO. **가림 없음** | 🔴 **별도** (numpy, CPU). **비교·검증용으로만** 남겨둠 — 기본 엔진은 SBR |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `drjit` | 1.3.1 | Mitsuba 의 JIT 컴파일러 — GPU 커널 생성 |
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `manifold3d` | 3.5.2 | **불리언(CSG) 엔진** — trimesh 백엔드. 겹친 파트의 내부 면을 녹여 없앤다 |
| `shapely` | 2.1.2 | 2D 단면 폴리곤(버퍼·오프셋) → 로프트 입력 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |
| `Pillow` | 12.2.0 | GIF 합성 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: GPU 1장 자동선택(현재 CUDA_VISIBLE_DEVICES=2), 전체 재현 약 15~20분. 가장 무거운 것은 SBR 마이크로도플러(자세 144개 × 5종, 자세마다 광선을 새로 쏜다).

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
# 전체 (그림 + Sionna 렌더 + SBR 마이크로도플러 + 노트북)
~/.venvs/py312/bin/python src/build_report1.py

# 부분만
~/.venvs/py312/bin/python src/viz_report1.py --only chamber,radiomap   # 챔버 + 라디오맵
~/.venvs/py312/bin/python src/viz_report1.py --only mesh,cad,gallery   # 메쉬 검증 + 갤러리
~/.venvs/py312/bin/python src/viz_report1.py --only art,md,gif         # 분절 + 마이크로도플러

# 하위 검증 도구를 직접
~/.venvs/py312/bin/python src/mesh_check.py      # 메쉬 전수검사(빌드 게이트)
~/.venvs/py312/bin/python src/materials.py       # 재질 표 (Sionna 에서 읽어온 값)
~/.venvs/py312/bin/python src/rcs_sbr.py         # SBR 해석해 검증(평판/금속구)
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report1.json` | 이 노트북의 **모든 숫자**의 출처 |
| `outputs/figures/report1_*.png` | 그림 7장 (matplotlib) |
| `outputs/renders/r1_*.png` | **Sionna 렌더** 25장 (1600×1100, spp=640) |
| `outputs/renders/r1_40_articulation.gif` | 분절 애니메이션 (Sionna 렌더 프레임) |
| `assets/meshes/drones/<key>/*.obj` | 부위별 드론 메쉬 (Sionna 가 읽는 바로 그 파일) |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **흡수체 재질은 실측치가 아니라 모델값이다.** `materials.py` 의 absorber(εr=1.4, σ=1.2)는 문헌 기반 모델이고, 평평한 단일면 수직입사 |Γ|=0.549(≈−5.2 dB)로 **전혀 낮지 않다**. 무반사(−25~−30 dB)는 **피라미드 골짜기의 다중반사**로 달성되는 기하 효과이고, 그 −25 dB 는 **설계 목표**이지 이 씬에서 측정한 값이 아니다.
- **공식 외형 0.00 % 일치는 '증거'가 아니라 '제약'이다.** 우리가 그렇게 맞췄다(`frame_fit_scale`). 진짜 검증은 Matrice 4E 대각선(-1.8 %)뿐이다.
- **내부 형상은 추정이다.** 배터리·PCB·모터를 '금속 상자/판/원통'으로 뒀다. 티어다운 사진 기반이지 CT 스캔이 아니다. RCS 의 절대값은 이 가정에 민감하다(report2 에서 다룬다).
- **호버 RPM 은 유도값이다.** $C_T$ 를 0.10~0.12 로 가정했다. Matrice 4E 는 적대적 검증이 $C_T$≈0.08~0.09 / 호버 4000~4500 rpm 을 권고했는데(공식 최대 7500 rpm 앵커), 우리는 아직 3800 rpm(=$C_T$ 0.108)을 쓰고 있다 — **미해결 불일치**.
- **마이크로도플러는 '미리보기'다.** 잡음·안테나패턴·전파손실이 없는 **자유공간 단일산란** 복소장이다. 실제 검출(패시브 파형·ECA·CFAR)은 report3 소관이다.
- **Sionna RT 는 표적 σ 를 주지 않는다** — 전파용 path solver 에는 **산란적분 단계가 없기 때문**이다. (레이트레이싱 일반이 RCS 를 못 낸다는 뜻이 **아니다**. SBR 이 바로 레이트레이싱이고, 그것으로 σ 를 계산한다.)

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| **report2** (OFDM 파형 + RCS) | 여기서 만든 **메쉬와 재질**을 그대로 받아 σ 를 낸다 |
| **report3** (Sionna RT 광선 반사) | 여기서 세운 **챔버**에서 광선을 쏜다 — 바닥 반사/유령이 §1 의 semi-anechoic 에서 나온다 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **RCS (σ)** | 레이더 반사 단면적 [m²]. '이 표적이 얼마나 밝게 되비추는가'. dBsm = 10·log₁₀(σ/1 m²) |
| **PO** | 물리광학(Physical Optics). 표면에 유도된 전류를 **적분**해 산란장을 구한다. σ 는 이 적분에서 나온다 |
| **GO** | 기하광학. 표면을 '국소 무한 거울'로 본다. 벽·바닥엔 정확하지만 **면적 항이 없어 σ 를 못 준다** |
| **SBR** | Shooting-and-Bouncing Rays. **광선(GO)으로 조명면을 찾고 그 위에서 PO 적분**. 상용 EM 솔버의 표준 방법 |
| **가림(occlusion)** | 앞의 면에 막혀 실제로는 안 보이는 면. 이걸 안 빼면 σ 를 과대평가한다 |
| **바이스태틱** | 송신기와 수신기가 **떨어져 있는** 레이더. 패시브 레이더가 이 형태 |
| **도플러** | 표적이 움직여 생기는 주파수 변화. 정지 물체(클러터)와 표적을 가르는 축 |
| **마이크로도플러** | 표적의 **부분 운동**(프로펠러 회전)이 만드는 도플러 미세구조. 드론의 지문 |
| **ECA** | 직접파(TX→RX 가시선)를 지워 표적을 드러내는 전처리. 지연된 기준신호 부분공간으로 사영해 뺀다 |
| **CFAR** | 일정 오경보율 검출기. 주변 셀의 잡음 수준을 보고 문턱을 정한다 |
| **semi-anechoic** | **벽·천장만** 흡수체이고 **바닥은 반사성**인 챔버. 우리 챔버가 이것 (anechoic 아님) |

</details>

---


## §1. 챔버 — Sionna 안의 실험장

**30 × 20 × 11 m, semi-anechoic.** 벽 4면 + 천장은 피라미드 흡수체(피치 60 cm, 높이 40 cm), **바닥은 반사성 콘크리트 타일**입니다. 메쉬는 `src/chamber.py` 가 만들고(삼각형 19,988개, 부위 16종), `src/scene_build.py` 가 부위별 OBJ → Sionna `SceneObject` 로 올립니다.

> ⚠️ **anechoic 이 아닙니다.** 6면 전부 흡수체여야 자유공간을 모사하는 anechoic 이고, 바닥만 반사면 정의상 **semi-anechoic (ground-plane) 챔버**입니다 — EMC 시험장의 표준 형태이기도 합니다. 이 한 가지가 report3 의 **바닥 반사/유령**으로 그대로 이어집니다.

![chamber](outputs/figures/report1_chamber_geometry.png)

### 재질 — **단일 진리원**

`src/materials.py` 가 정의한 표 **하나**를 Sionna RT(전파)와 우리 PO/SBR(RCS)이 **똑같이** 읽습니다. 예전엔 표가 두 개였고 조용히 어긋나 있었습니다(짐벌 카메라: Sionna 는 plastic |Γ|=0.244, PO 는 0.85 — **10.9 dB** 차이).

원칙은 **ITU 가 있으면 ITU 를 쓴다**입니다. metal·concrete 의 (εr, σ)는 우리가 적지 않고 **Sionna 에게 물어봅니다**(`ITURadioMaterial` → 주파수 자동 보정). ITU 에 없는 것(흡수체·플라스틱·카본)만 커스텀입니다.

| 재질 | 출처 | eps_r | sigma [S/m] | S | \|Γ\| 벌크 | \|Γ\| PO/SBR |
|---|---|---|---|---|---|---|
| `metal` | **ITU** (metal) | 1.00 | 1e+07 | 0.00 | 1.000 | 1.00 |
| `camera_assembly` | **ITU** (metal) | 1.00 | 1e+07 | 0.00 | 1.000 | 0.85 |
| `pcb` | **ITU** (metal) | 1.00 | 1e+07 | 0.00 | 1.000 | 0.80 |
| `concrete_light` | **ITU** (concrete) | 5.24 | 0.123 | 0.00 | 0.395 | 0.39 |
| `concrete_dark` | **ITU** (concrete) | 5.24 | 0.123 | 0.00 | 0.395 | 0.39 |
| `plastic` | custom | 2.70 | 0.02 | 0.20 | 0.244 | 0.28 |
| `plastic_blue` | custom | 2.70 | 0.02 | 0.20 | 0.244 | 0.28 |
| `prop_plastic` | custom | 2.70 | 0.02 | 0.20 | 0.244 | 0.25 |
| `carbon` | custom | 5.00 | 3e+03 | 0.30 | 0.989 | 0.90 |
| `absorber` | custom | 1.40 | 1.2 | 0.35 | 0.549 | 0.55 |

**ITU 재질의 산란계수 S 는 전부 0 입니다** (순수 정반사). 이것이 뒤에서 중요해집니다: 모터·배터리·PCB 는 ITU metal 이라 **확산 산란 기여가 0** 이고, 그래서 Sionna 의 확산 경로로는 표적 σ 가 창발하지 않습니다. σ 는 **적분**에서 나옵니다 → 그래서 SBR(광선 + PO 적분)을 씁니다.

![materials](outputs/figures/report1_materials.png)

### Sionna 가 직접 그린 챔버

아래는 matplotlib 도식이 아니라 **Sionna RT 의 렌더러**(`Scene.render_to_file`, Mitsuba 3 경로추적, GPU)가 그린 그림입니다 — 1600×1100, spp=640. 즉 **우리가 상상한 방이 아니라 시뮬레이터가 실제로 보고 있는 방**입니다.

| 외관 (금속 차폐 + 강철 골조) | 내부 (앞벽 제거) |
|---|---|
| ![ext](outputs/renders/r1_10_chamber_exterior.png) | ![wide](outputs/renders/r1_11_chamber_wide.png) |
| **위에서** (`clip_at`=9 m 로 천장을 잘라냄) | **바닥 스침** (grazing) |
| ![top](outputs/renders/r1_12_chamber_top.png) | ![graze](outputs/renders/r1_13_chamber_grazing.png) |
| **측면** — 바닥 반사 삼각형이 보인다 | **표적 상공** |
| ![side](outputs/renders/r1_14_chamber_side.png) | ![ot](outputs/renders/r1_15_chamber_over_target.png) |

### 라디오맵 — 그리고 **드론의 그림자**

`sionna.rt.RadioMapSolver` 로 TX 한 대가 만드는 전파 세기 분포를 두 평면에서 계산했습니다(12 M rays/TX, 셀 15 cm, max_depth=3). 

**바닥면**을 보면 흡수체 벽 쪽은 어둡고 **바닥이 밝습니다** — semi-anechoic 이 그림으로 나온 것입니다. **드론 평면**에서는 드론이 TX 광선을 막아 **자기 뒤로 그림자를 드리웁니다**. 이 그림자가 바로 '표적이 전파를 실제로 가로챈다'는 증거이고, SBR 이 계산하는 **가림(occlusion)** 과 같은 물리입니다.

| 바닥면 (z = 0.05 m) | 드론 평면 (z = 5.5 m) — 그림자 |
|---|---|
| ![rmf](outputs/renders/r1_20_radiomap_floor_top.png) | ![rmd](outputs/renders/r1_20_radiomap_droneplane_top.png) |
| ![rmf2](outputs/renders/r1_21_radiomap_floor_wide.png) | ![rmd2](outputs/renders/r1_21_radiomap_droneplane_wide.png) |

## §2. 드론 메쉬 — **어떻게 만들었나**

### 2.1 먼저 자료를 모았고, 그다음 **그 자료를 공격했다**

5종(Mini 5 Pro / Mavic 4 Pro / Matrice 4E / S1000+ / Phantom 4)을 심층조사했습니다 — DJI 공식 스펙 페이지, **사용자 매뉴얼 PDF 를 직접 내려받아 PyMuPDF 로 텍스트 추출**, 티어다운 문서. 결과는 `docs/drone_specs_2026.json` 에 **기종별 출처 URL 과 함께** 들어 있습니다.

그리고 **적대적 검증**을 했습니다: 다른 에이전트가 그 조사를 **'틀렸다고 가정하고'** 1차 소스로 재확인했습니다. **지어낸 숫자 3건**이 나왔습니다.

| # | 기종 | 조사가 주장한 것 | 실제 | 왜 위험했나 |
|---|---|---|---|---|
| 1 | mavic4pro | "Mavic 3 호버 4,000~4,700 rpm **실측 보고**" | **그런 보고는 없다** (출처 0) | 호버 RPM 앵커 — 마이크로도플러 전체가 여기 매달린다 |
| 2 | matrice4e | "매뉴얼 C2 인증표: **최대 6,130 RPM**, 음향파워 85 dB" (직접 인용 형식) | 그 페이지엔 **7,500 RPM / 82 dB** | **존재하지 않는 인용문**. k_T·호버·팁속도·최대도플러가 전부 이 값에 매달려 있었다 |
| 3 | phantom4 | 공식 외형 289.5 × 289.5 × 196 mm 를 **안 쓰고 있었다** | DJI Quick Start Guide 에 있다 | 있는 공식값을 두고 추정하고 있었다 |

> 🔑 2번이 가장 위험합니다. **출처를 지정한 직접 인용 형식**이었고, `unknown` 필드에서 "공식값, 지어낸 값 아님"이라고 **방어까지** 하고 있었습니다. 그럴듯한 숫자 하나가 파이프라인 전체를 오염시킵니다. → 그래서 이 리포트의 규칙은 **"모르는 건 모른다고 쓴다"** 입니다.

### 2.2 CAD — 왜 자작 `geom.py` 를 버렸나

예전 메쉬는 프리미티브(원기둥·다각기둥)를 **겹쳐 쌓은** 것이었습니다. 문제는 미관이 아니라 **물리**였습니다:

- 동체가 각진 다각기둥 (실물은 매끈한 눈물방울) → 투영면적/실루엣이 틀리면 **σ 가 틀린다**
- 짐벌이 상자 (Mavic 4 는 **구형 Infinity 짐벌**)
- **불리언이 없어** 겹친 파트의 **내부 면이 그대로 남았다** ← 이게 진짜 버그

그래서 **라이브러리를 쓰기로** 했습니다 — `src/cadkit.py`:

| 라이브러리 | 무엇을 |
|---|---|
| **trimesh** | 메쉬 자료구조 · 로프트/스윕/회전체 · 스무딩 · **검증** |
| **manifold3d** | **불리언(CSG)** 엔진 (trimesh 백엔드) — 겹친 파트를 하나의 껍질로 녹인다 |
| **shapely** | 2D 단면 폴리곤 (버퍼·오프셋·회전) → 로프트 입력 |
| **scipy** | 스플라인(단면 보간·암 경로) |

![cad](outputs/figures/report1_cad_pipeline.png)

**측정된 대가**: 불리언을 끄면 mavic4pro 프레임에서 겹친 파트의 **내부 면 1374개**(면적의 **12.9 %**)가 드론 **속에** 살아남습니다. PO 는 가림 판정이 없으므로 그것을 그대로 적분하고, σ 가 **+0.77 dB** 부풀려집니다 (-18.6 → -19.3 dBsm, 방위 60개 평균 @ el=15°).

삼각형 수는 오히려 비슷합니다(13,432 → 13,484) — CSG 는 내부를 지우는 대신 교차선을 새로 만듭니다. **이것은 최적화가 아니라 정확도 수정입니다.**

### 2.3 검증 — trimesh 가 실제로 잡아낸 버그

`src/mesh_check.py` 는 드론을 그룹 → **연결요소(부품)** 로 쪼개 trimesh 에게 묻습니다: watertight 인가 / winding 이 일관적인가 / 법선이 바깥을 보는가 / 퇴화면이 있는가. **5/5 통과**.

왜 미관 문제가 아닌가: PO 도 SBR 도 **`n̂·û > 0` 의 부호**로 '이 면이 조명되는가'를 판정합니다. 뒤집힌 캡이나 면적 0 삼각형은 그 판정을 **조용히** 오염시킵니다. 그리고 **프로펠러는 마이크로도플러 신호 그 자체**입니다.

실제로 잡은 것 2건:
1. **`revolve()` 의 r=0 링을 apex 로 안 접었다** → 같은 자리에 정점이 seg개씩 쌓여 **퇴화 삼각형 512개** (모터 288 + 프로펠러 224). 퇴화면은 법선이 정의되지 않는다.
2. **옛 `prop_blade()` 의 캡 2장이 뒤집혀 있었다** (법선 안쪽). **코드 주석에는 "outward" 라고 적혀 있었다.**

![meshcheck](outputs/figures/report1_meshcheck.png)

### 2.4 공식 외형 정합 — 그리고 **방법론이 스스로를 검증한 순간**

| 기종 | 이름 | DJI 공식 L×W×H [mm] | 우리 메쉬 [mm] | 오차 [%] | 대각선(메쉬) [mm] | 프롭 [mm]×엽 | 삼각형 |
|---|---|---|---|---|---|---|---|
| `mini5pro` | DJI Mini 5 Pro | 255.0 × 181.0 × 91.0 | 255.0 × 181.0 × 91.0 | +0.00 / +0.00 / +0.00 | 255 | 152 × 2 | 24,398 |
| `mavic4pro` | DJI Mavic 4 Pro | 328.7 × 390.5 × 135.2 | 328.7 × 390.5 × 135.2 | +0.00 / +0.00 / +0.00 | 439 | 267 × 2 | 25,676 |
| `matrice4e` | DJI Matrice 4E | 307.0 × 387.5 × 149.5 | 307.0 × 387.5 × 149.5 | +0.00 / +0.00 / +0.00 | 431 | 274 × 2 | 28,714 |
| `s1000plus` | DJI S1000+ | 1016.0 × 1016.0 × 380.0 | 1016.0 × 1016.0 × 380.0 | +0.00 / +0.00 / +0.00 | 1044 | 381 × 2 | 39,158 |
| `phantom4` | DJI Phantom 4 | 289.5 × 289.5 × 196.0 | 289.5 × 289.5 × 196.0 | +0.00 / +0.00 / +0.00 | 357 | 240 × 2 | 27,806 |

**0.00 % 는 증거가 아닙니다 — 제약입니다.** `drones.frame_fit_scale()` 이 축별 배율로 그렇게 맞춥니다. 이 표가 증명하는 것은 '우리가 스펙시트를 지켰다'뿐입니다.

**진짜 검증은 대각선입니다.**

- **Mavic 4 Pro**: DJI 는 대각선을 **공개한 적이 없습니다**. 떠돌던 '400 mm'는 **출처가 없고**, 공식 외형(328.7 × 390.5 mm)과 267 mm 프롭 앞에서 **기하학적으로 불가능**합니다 (400 mm 면 앞뒤 모터 간격 ≈ 253 mm < 프롭 267 mm → 디스크가 14 mm 겹친다. 비중첩 쿼드에선 불가). 외형에서 유도하면 **439 mm**, 적대적 검증도 독립적으로 **~440 mm** 라고 확인했습니다.
- **Matrice 4E**: DJI 가 **외형과 대각선(438.8 mm)을 둘 다** 공개한 유일 기종입니다. 우리는 **외형만** 먹여 메쉬를 맞췄는데, 나온 대각선이 **431 mm (-1.8 %)** 입니다. → 대각선을 한 번도 보지 않고 1.8 % 안에 맞췄다는 뜻이고, 이것이 **방법론의 독립 검증**입니다.

![envelope](outputs/figures/report1_envelope.png)

### 2.5 Sionna 렌더 갤러리 — 5종 × (iso / side / top)

**side 뷰를 보십시오.** 짐벌 실루엣(Mavic 4 의 구형 Infinity 볼, Matrice 의 측량 페이로드 + RTK 돔, Phantom 의 일체형 스키드, S1000+ 의 긴 다리)과 **높이**가 여기서 갈립니다. 이 메쉬가 **SBR 이 실제로 적분하는 바로 그 메쉬**입니다.

| 기종 | iso | side | top |
|---|---|---|---|
| **DJI Mini 5 Pro**<br>4로터 · 250 g · 24,398 tris | ![](outputs/renders/r1_30_drone_mini5pro_iso.png) | ![](outputs/renders/r1_30_drone_mini5pro_side.png) | ![](outputs/renders/r1_30_drone_mini5pro_top.png) |
| **DJI Mavic 4 Pro**<br>4로터 · 1063 g · 25,676 tris | ![](outputs/renders/r1_30_drone_mavic4pro_iso.png) | ![](outputs/renders/r1_30_drone_mavic4pro_side.png) | ![](outputs/renders/r1_30_drone_mavic4pro_top.png) |
| **DJI Matrice 4E**<br>4로터 · 1219 g · 28,714 tris | ![](outputs/renders/r1_30_drone_matrice4e_iso.png) | ![](outputs/renders/r1_30_drone_matrice4e_side.png) | ![](outputs/renders/r1_30_drone_matrice4e_top.png) |
| **DJI S1000+**<br>8로터 · 9500 g · 39,158 tris | ![](outputs/renders/r1_30_drone_s1000plus_iso.png) | ![](outputs/renders/r1_30_drone_s1000plus_side.png) | ![](outputs/renders/r1_30_drone_s1000plus_top.png) |
| **DJI Phantom 4**<br>4로터 · 1380 g · 27,806 tris | ![](outputs/renders/r1_30_drone_phantom4_iso.png) | ![](outputs/renders/r1_30_drone_phantom4_side.png) | ![](outputs/renders/r1_30_drone_phantom4_top.png) |

## §3. 분절(articulation) — 마이크로도플러와 Gazebo 를 위해

### 3.1 몸체 자세 ⟂ 로터 스핀

`drones.pose_articulated(spec, body_rpy, body_pos, rotor_phase_deg)` 는 **두 회전을 분리**해 적용합니다:
- 프레임(비회전부) ← 몸체 변환 **B** 만
- 로터 k 의 프로펠러 ← **B ∘ (로터 위치) ∘ (장착 오프셋 + 스핀 위상 θ_k)**

**검증(코드 출력)**:
- 블레이드만 90° 스핀시켰을 때 **프레임 정점의 최대 이동 = 0.00e+00 m** (= 0, 완전 분리)
- `pose_articulated(스핀 0)` 의 출력이 기존 `build_drone()` 과 **완전히 동일**: True (→ report2 의 RCS 가 바뀌지 않는다)
- RPY(0°,12°,25°) + 로터별 위상(0/45/90/135°) 동시 적용 시 정점 12,880/12,880 개가 움직인다

**이 분해가 두 곳에서 동시에 필요합니다**: 마이크로도플러의 '개별 회전 강체'와 Gazebo 의 'link + revolute joint' 는 **같은 것**입니다.

![articulation](outputs/renders/r1_40_articulation.gif)

*(Sionna 렌더러가 프레임마다 그린 GIF — 로터 위상을 0→180° 돌리면서 몸체도 살짝 흔들었습니다. 몸체 자세와 블레이드 회전이 **독립**임을 눈으로 볼 수 있습니다.)*

### 3.2 호버 RPM — **가정이 아니라 유도**

DJI 는 호버 RPM 을 공개하지 않습니다. 그래서 **물리로 유도**합니다:

$$T = C_T\,\rho\,n^2 D^4 \quad\Rightarrow\quad n = \sqrt{\frac{m g / N_{rotor}}{C_T\,\rho\,D^4}}$$

여기서 $C_T$ 는 프로펠러 추력계수(소형 멀티로터 전형 **0.10~0.12**), $\rho$=1.225 kg/m³, $D$=프롭 지름, $n$=회전수[rev/s]. 호버는 총추력 = 중량이므로 로터당 추력 $T = mg/N$ 입니다.

| 기종 | 질량 [g] | 로터 | T/로터 [N] | D [mm] | $C_T$=0.12~0.10 → rpm | **모델값 rpm** | 함의 $C_T$ | f_rot [Hz] | **flash [Hz]** | v_tip [m/s] | **f_tip [Hz]** |
|---|---|---|---|---|---|---|---|---|---|---|---|
| `mini5pro` | 250 | 4 | 0.61 | 152 | 5274 – 5777 | **5500** | 0.110 | 91.7 | **183** | 44 (M0.13) | **990** |
| `mavic4pro` | 1063 | 4 | 2.61 | 267 | 3544 – 3882 | **3600** | 0.116 | 60.0 | **120** | 50 (M0.15) | **1135** |
| `matrice4e` | 1219 | 4 | 2.99 | 274 | 3603 – 3947 | **3800** | 0.108 | 63.3 | **127** | 55 (M0.16) | **1230** |
| `s1000plus` | 9500 | 8 | 11.65 | 381 | 3679 – 4030 | **3600** | 0.125 | 60.0 | **120** | 72 (M0.21) | **1620** |
| `phantom4` | 1380 | 4 | 3.38 | 240 | 4997 – 5474 | **5500** | 0.099 | 91.7 | **183** | 69 (M0.20) | **1559** |

*(flash = blades × rpm/60,  f_tip = 2·v_tip/λ·cos(el),  λ = 8.57 cm @ 3.5 GHz, el = 15°)*

**모델이 쓰는 값이 전부 물리 밴드 안에 들어옵니다** — 즉 이 rpm 들은 지어낸 게 아니라 유도된 것입니다.

> ### 🔴 정정: mavic4pro 5500 → **3600 rpm**
> 옛 5500 rpm 은 **호버가 아니었습니다.** 그 회전수는 중량의 **2.3 배** 추력을 냅니다 — 즉 **최대추력 회전수**(DJI 공식 max 6000 rpm)에 가까운 값이었습니다.
> - flash: 183 → **120 Hz**
> - f_tip: 1734 → **1135 Hz**
> **옛 숫자(183 Hz / 1734 Hz)를 쓰지 마십시오.**

**flash 를 정확히 세는 법** (자주 틀리는 곳): 2엽 프로펠러는 **180° 대칭**이라 **프로펠러가** 1회전에 브로드사이드를 **2번** 보여줍니다 → flash = 60 rev/s × 2 = **120 Hz**. 여기에 블레이드 수를 **한 번 더** 곱하면 240 Hz 가 되는데 이건 틀립니다.

![hover](outputs/figures/report1_hover_rpm.png)

### 3.3 마이크로도플러 미리보기 — **SBR(가림 포함)**

`microdoppler.microdoppler_sbr()` 로 5종의 슬로타임 복소장 E(t) 를 냈습니다. **공식이 아닙니다** — 로터 위상마다 메쉬를 다시 포즈시키고 **Mitsuba 가 광선을 새로 쏩니다**. 그래서 블레이드가 동체 뒤로 돌아가면 **정말로 산란을 멈춥니다**(가림).

핵심 트릭: 드론 전체 자세는 **단일 각도 φ = ωt 의 함수**이고, n엽 프로펠러는 **360/n 도마다 반복**합니다. → φ ∈ [0, 180°) 를 144개로 잘라 SBR 을 **선계산**하고, 시간축은 조회+보간합니다. 시간 스텝마다 광선을 쏘지 않습니다.

| 기종 | rpm | flash [Hz] | f_tip [Hz] | \|DC\|/std(AC) — 순수 PO | **SBR (가림)** | 차이 |
|---|---|---|---|---|---|---|
| `mini5pro` | 5500 | 183 | 990 | 241.7 (+47.7 dB) | **4.2** (+12.4 dB) | **+35.3 dB** |
| `mavic4pro` | 3600 | 120 | 1135 | 291.1 (+49.3 dB) | **4.8** (+13.6 dB) | **+35.7 dB** |
| `matrice4e` | 3800 | 127 | 1230 | 95.9 (+39.6 dB) | **11.6** (+21.3 dB) | **+18.4 dB** |
| `s1000plus` | 3600 | 120 | 1620 | 22.9 (+27.2 dB) | **9.2** (+19.2 dB) | **+8.0 dB** |
| `phantom4` | 5500 | 183 | 1559 | 284.3 (+49.1 dB) | **4.3** (+12.6 dB) | **+36.5 dB** |

**가림을 넣으면 정적 몸체 받침대(DC)가 내려가고 블레이드 선(AC)이 상대적으로 올라옵니다 — 8~36 dB.** 옛 순수 PO 그림은 (a) 동체 뒤에 숨은 블레이드와 (b) 셸 속에 봉인된 배터리·PCB 를 **계속 세고 있었습니다**. 즉 **마이크로도플러 검출은 옛 PO 추정보다 그만큼 쉽습니다.**

![microdoppler](outputs/figures/report1_microdoppler.png)

> ⚠️ 이건 **미리보기**입니다: 잡음·안테나패턴·경로손실이 없는 자유공간 단일산란 복소장입니다. 그리고 **PRF 가 20 kHz** 입니다 — 실제 패시브 파형의 파일럿 반복률(LTE CRS 1 kHz, 5G SSB 50 Hz)로는 f_tip(1135 Hz)이 **접힙니다**. 그 문제는 report3 소관입니다.

### 3.4 Gazebo / PX4 로 가려면 무엇이 더 필요한가

**기하는 이미 됐습니다**: Gazebo 는 로터마다 **별도 link + revolute joint** 를 요구하는데, 그건 마이크로도플러가 요구하는 '개별 회전 강체'와 **정확히 같습니다**. 모델 하나가 둘 다를 떠받칩니다.

**없는 것은 동역학입니다.** `docs/drone_specs_2026.json` 의 `gazebo_px4` 필드에 조사 결과가 있고, **공개된 것과 추정해야 하는 것**은 이렇게 갈립니다:

![gazebo](outputs/figures/report1_gazebo.png)

| 파라미터 | 상태 | 비고 |
|---|---|---|
| 질량 m | 🟢 **공개** | 5종 전부 (S1000+ 는 TOW 9.5 kg — 기체중량 4.4 kg 과 구분할 것) |
| 프롭 지름·엽수·피치 | 🟢 **공개** | DJI 부품 페이지 |
| 최대 프롭 RPM | 🟢 공개 (mini 7800 / matrice 7500 / phantom 8500) | mavic·s1000 은 미공개 → 역산 |
| 추력계수 k_T | 🟡 **유도** | $k_T = T/\omega^2$. 예: mini5pro **1.85e-6**, phantom4 **1.02e-5**, s1000+ **8.2e-5** N·s²/rad² |
| 관성모멘트 Ixx/Iyy/Izz | 🔴 **미공개 — 추정** | 점질량 분해. 예: mini5pro Ixx 9.6e-4 > Iyy 5.5e-4 — **좌우로 넓은 배치라 롤 관성이 더 크다** |
| 모멘트계수 k_M | 🔴 **미공개 — 추정** | k_M/k_T ≈ 0.015~0.020 m (전형값) |
| 모터 시상수 τ | 🔴 **미공개 — 추정** | 0.02~0.05 s. **스텝응답 측정 필요** |

> **정직하게**: 관성텐서·k_M·시상수는 **어느 기종도 DJI 가 공개하지 않습니다.** PX4 SITL 을 붙이려면 이 3개를 추정하거나 **재야 합니다**. 그리고 Matrice 4E 의 k_T(1.9e-5)는 **위조된 최대 RPM(6130)에 앵커링돼 있었습니다** — 적대적 검증은 공식 7500 rpm 기준으로 $C_T$≈0.08~0.09, k_T≈1.4~1.6e-5, 호버 4000~4500 rpm 을 권고합니다. 우리는 아직 3800 rpm 을 쓰고 있습니다 — **미해결**.

> **역으로 좋은 소식**: 모터 추력계수와 질량이 정해지면 호버 rpm 은 **가정이 아니라 유도값**이 됩니다(이 리포트가 이미 그렇게 했습니다). PX4/Gazebo 모델이 완성되면 f_tip·flash 는 **비행 시뮬에서 떨어져 나오는 값**으로 승격됩니다.

## §4. 정리 & 다음

### 답한 질문: *Sionna 안에 무엇을 어떻게 세웠고, 그 드론 모델을 믿어도 되는가?*

**세운 것**
- **챔버**: 30×20×11 m semi-anechoic (삼각형 19,988, 부위 16종) → Sionna Scene. 재질은 **단일 진리원**(ITU 우선).
- **드론 5종**: trimesh + manifold3d CAD, 삼각형 24,398~39,158, 부위별 OBJ → Sionna RadioMaterial.
- **분절**: 몸체 자세 ⟂ 로터 스핀. 마이크로도플러와 Gazebo 의 **공통 토대**.

**믿어도 되는 근거 (측정)**
| 주장 | 근거 | 값 |
|---|---|---|
| 외형이 실물과 같다 | DJI 공식 L×W×H 대조 | **0.00 %** (5/5) — 단, 이건 **제약**이다 |
| 방법론이 맞다 | Matrice 4E 대각선(우리가 안 먹인 값) | **431 mm vs 공식 438.8 (-1.8 %)** |
| 메쉬가 깨끗하다 | trimesh 전수검사 (watertight/winding/법선/퇴화면) | **5/5 통과** (버그 2건 잡고 고침) |
| 불리언이 필요하다 | 내부 면 → PO 편향 측정 | **1374면 = +0.77 dB** |
| 호버 rpm 이 물리적이다 | $T=C_T\rho n^2D^4$ 밴드 | 5종 전부 $C_T$ 0.10~0.12 안 |
| 분절이 진짜다 | 블레이드 90° 스핀 시 프레임 이동 | **0.0e+00 m** |

**믿으면 안 되는 것** (앞머리 8️⃣ 참조): 흡수체 −25 dB(설계목표, 미측정) · 내부 형상(추정) · Matrice 4E 호버 rpm(미해결 불일치) · 마이크로도플러는 잡음 없는 미리보기.

### 다음
| 리포트 | 무엇을 이어받나 |
|---|---|
| **report2** — OFDM 파형 + RCS | 이 **메쉬와 재질**을 그대로 받아 SBR 로 σ 를 낸다. 점유모드 G1 포함. |
| **report3** — Sionna RT 광선 반사 | 이 **챔버**에서 광선을 쏜다. §1 의 **반사성 바닥**이 거기서 유령이 된다. |